In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import requests
from socket import gethostname, gethostbyname
from IPython import get_ipython

# --- GLOBALS & CONFIGURATION ---
VALID_PROTEINS = {
    "ASCT2", "CCR5", "CGRPR", "FZD7", "LAT1", "MCT1",
    "MurJ", "PTH1R", "PfMATE", "SERT", "STP10", "ZnT8"
}

# Stores the final command for later execution
FULL_PREDICT_COMMAND = ""

def print_documentation():
    """Prints the strict formatting rules for the notebook title."""
    doc = """


TODO: manually adjust filepath to target location under point '# 6) Construct
the Bash execution command'
================================================================================
DYNAMIC COMMAND GENERATOR - FORMATTING RULES
================================================================================
This script automatically builds the execution command based on the title
of this Colab notebook. To work correctly, the filename must adhere to
the following rules:

1. Base Prefix:
   The title should ideally start with 'colab_predict_' (this prefix is
   automatically stripped from the final job name).

2. Ablation Parameters (Can appear in ANY order):
   - depth_[X]: Sets --max-msa. [X] must be an EVEN integer or 'None'.
     (Note: depth_5120 automatically sets 512:5120; others set X/2:X).
   - row_mask_[X]: Sets row_masking n_keep. [X] must be a number or 'None'.
   - col_mask_[X]: Sets row_or_col_masking frac. [X] is a percentage (10 = 0.1).
   - qu_mask_[X]: Sets query_masking frac. [X] is a percentage.

   * Note: If a parameter is entirely missing from the title, the script
     will automatically infer 'None' for that parameter.

3. Target Proteins:
   Must contain one or more of the following valid proteins, separated by '_':
   ASCT2, CCR5, CGRPR, FZD7, LAT1, MCT1, MurJ, PTH1R, PfMATE, SERT, STP10, ZnT8

4. Job Name & Custom Suffixes:
   Any text in the title (after 'colab_predict_') that is NOT an ablation
   parameter or a recognized protein will remain as the base string for
   the --job name. (e.g., adding '_test' will include it in the job name).

5. Ignored Comments:
   Anything following '___' (3x '_') in the filename is ignored and excluded from
   both the job name and parameters (e.g., '...---RERUN_after_fail.ipynb').
================================================================================
"""
    print(doc)

def execute_dynamic_prediction_job():
    global FULL_PREDICT_COMMAND

    # print_documentation()

    # 1) Programmatically fetch the current notebook file name
    try:
        ip = gethostbyname(gethostname())
        session_info = requests.get(f"http://{ip}:9000/api/sessions").json()
        filename = session_info[0]["name"]
        print(f"Successfully detected notebook title: {filename}")
    except Exception as e:
        print(f"Warning: Could not automatically detect filename: {e}")
        print("Falling back to manual file name string assignment...")
        filename = "colab_predict_depth_32_row_mask_50_col_mask_10_qu_mask_15_MurJ_PfMATE_ZnT8_test.ipynb"

    # 2) Clean the filename (Remove comments and extension)
    clean_name = filename.replace(".ipynb", "").split("___")[0]

    # 3) Extract base job title
    job_title = clean_name
    if job_title.startswith("colab_predict_"):
        job_title = job_title[len("colab_predict_"):]

    working_str = job_title
    params = {}

    # 4) Extract ablation parameters and depths (agnostic to order)
    for param in ["depth", "row_mask", "col_mask", "qu_mask"]:
        match = re.search(rf"(?:^|_){param}_([^_]+)(?=_|$)", working_str)

        if match:
            val = match.group(1)

            # Strict Validation: Must be a number or 'None'/'none'
            if val.lower() != 'none':
                try:
                    num_val = float(val)
                except ValueError:
                    raise ValueError(f"Error: Invalid value '{val}' for '{param}'. Expected a number or 'None'.")

                # Depth-specific rule: must be an even integer
                if param == "depth":
                    if not num_val.is_integer() or int(num_val) % 2 != 0:
                        raise ValueError(f"Error: Depth must be an even number. Got: {val}")

            params[param] = val.lower() if val.lower() == 'none' else val

            # Remove the extracted parameter and value from the working string
            working_str = working_str.replace(f"{param}_{val}", "")
            working_str = re.sub(r'_+', '_', working_str).strip('_')
        else:
            # Missing parameter infers "None"
            params[param] = "none"

    # 5) Extract Proteins & Identify Unrecognized Terms
    tokens = working_str.split('_') if working_str else []
    input_proteins = []
    unrecognized_terms = []

    for t in tokens:
        if not t: # Skip any empty strings from duplicate underscores
            continue
        if t in VALID_PROTEINS:
            input_proteins.append(t)
        else:
            unrecognized_terms.append(t)

    if not input_proteins:
        raise ValueError("Error: No valid protein names found in the filename.")

    # Trigger a warning if there are leftovers (which could be typos or intentional suffixes)
    if unrecognized_terms:
        print(f"----------- WARNING: The following terms were not recognized as valid proteins "
              f"and are being treated as custom job suffixes: -----------\n{', '.join(unrecognized_terms)}")

    # 6) Construct the Bash execution command
    cmd_parts = [
        f"uv run scripts/predict.py",
        f"--job {job_title}",
        f"--input {' '.join(input_proteins)}",
        f"--num-models 5",
        f"--num-seeds 5",
        # f'--drive "/content/drive/MyDrive/AlphaFold2 Ablation Study/04_Results"'
        f'--drive "/content/drive/MyDrive/AlphaFold2_Ablations_Output"'
    ]

    # --- Depth Argument ---
    if params["depth"] != "none":
        d_int = int(float(params["depth"]))
        if d_int == 5120:
            cmd_parts.append("--max-msa 512:5120")
        else:
            cmd_parts.append(f"--max-msa {d_int // 2}:{d_int}")

    # --- Row Mask Argument ---
    if params["row_mask"] != "none":
        cmd_parts.append(f'--ablation "row_masking:n_keep={params["row_mask"]}"')

    # --- Col Mask Argument ---
    if params["col_mask"] != "none":
        frac = float(params["col_mask"]) / 100.0
        cmd_parts.append(f'--ablation "row_or_col_masking:row_or_col=column,frac={frac},seed=0"')

    # --- Query Mask Argument ---
    if params["qu_mask"] != "none":
        frac = float(params["qu_mask"]) / 100.0
        cmd_parts.append(f'--ablation "query_masking:frac={frac},seed=7"')

    # Save to global variable for the next cell
    FULL_PREDICT_COMMAND = " ".join(cmd_parts)
    print_friendly_command = " \\\n  ".join(cmd_parts)

    print("\nFormulated command:")
    print(f"! {print_friendly_command}\n")
    print("Initialization complete. Run the execution cell when ready.")

# Run the formulation
execute_dynamic_prediction_job()

In [ ]:
! nvidia-smi

In [ ]:
! git clone https://github.com/samzirbo/alphafold2-ablation-study
%cd alphafold2-ablation-study

In [ ]:
! git checkout franc/feat/queryMasking-and-ablation-script-support

In [ ]:
! uv sync

In [9]:
# example for running colabfold ablations using a custom command:

# ! uv run scripts/predict.py --job depth_32_row_mask_50_col_mask_10_qu_mask_15_MurJ_test \
#   --input MurJ PfMATE ZnT8 --num-models 5 --num-seeds 5 --drive "/content/drive/MyDrive/AlphaFold2 Ablation Study/04_Results/testF" \
#   --max-msa 16:32 \
#   --ablation row_masking:n_keep=50 \
#   --ablation row_or_col_masking:row_or_col=column,frac=0.1,seed=0 \
#   --ablation query_masking:frac=0.15,seed=7

In [ ]:
if FULL_PREDICT_COMMAND:
    print("Executing dynamically generated prediction command...")
    get_ipython().system(FULL_PREDICT_COMMAND)
else:
    print("Warning: Command is empty. Please ensure you ran the initialization cell first.")